# SoulIllusions Kaggle GPU Backend

**Settings -> Accelerator -> GPU T4 x2 -> Internet ON**

Run the cell below. Copy the URL it prints into SoulIllusions.

Or use `py kaggle_auto.py` from your machine to run this automatically.

In [ ]:
# ==================== SOULILLUSIONS KAGGLE GPU BACKEND ====================
import os, sys, time, uuid, gc, json, traceback, threading, subprocess, re, asyncio
from pathlib import Path
from typing import Optional
import numpy as np

# --- Step 0: Verify GPU is available BEFORE any pip installs ---
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print(f"[0/6] GPU OK: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[0/6] WARNING: No GPU detected! Running in CPU mode (slower).")
    print("  To enable GPU: Kaggle.com -> Your notebook -> Settings -> Accelerator -> GPU T4 x2")

# --- Step 1: Install deps (preserve Kaggle CUDA torch) ---
print('[1/6] Installing dependencies...')
# Non-torch deps: safe to install normally
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'websockets',
    'imageio', 'imageio-ffmpeg', 'numpy',
    'safetensors', 'omegaconf', 'einops',
    'pydantic', 'aiofiles'
], capture_output=True)
# Torch-dependent deps: use --no-deps to avoid overwriting CUDA torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'diffusers', 'transformers', 'accelerate', 'huggingface_hub'
], capture_output=True)
# Cloudflared for tunnel
# Download cloudflared
import urllib.request
try:
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/usr/local/bin/cloudflared')
    os.system('chmod +x /usr/local/bin/cloudflared')
    print('  cloudflared installed OK')
except Exception as e:
    print(f'  cloudflared download failed: {e}')
    # Try curl as fallback
    os.system('curl -L -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 2>/dev/null')
    os.system('chmod +x /usr/local/bin/cloudflared')

# Verify torch still has CUDA after installs
if not HAS_GPU and not torch.cuda.is_available():
    print("  WARNING: Still no CUDA after installs.")
elif torch.cuda.is_available():
    HAS_GPU = True
    print(f"  Deps installed. GPU now available: {torch.cuda.get_device_name(0)}")

from fastapi import FastAPI, BackgroundTasks, WebSocket, WebSocketDisconnect, Request
from fastapi.responses import JSONResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

# --- Step 2: Backend code ---
print('[2/6] Loading backend code...')

OUTPUT_DIR = '/kaggle/working/outputs'
IMG_DIR = os.path.join(OUTPUT_DIR, 'images')
VID_DIR = os.path.join(OUTPUT_DIR, 'videos')
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(VID_DIR, exist_ok=True)

TERMINAL_TOKEN = uuid.uuid4().hex

app = FastAPI(title='SoulIllusions GPU Backend')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

image_status = {}
generation_status = {}
_pipes = {}

STYLE_MODIFIERS = {
    'cinematic': {'camera': 'cinematic camera, slow dolly push-in, shallow depth of field', 'lighting': 'dramatic lighting, golden hour, high contrast, film still', 'quality': 'movie quality, 4k, highly detailed, professional cinematography', 'negative': 'static shot, flat lighting, low contrast, amateur, blurry'},
    'realistic': {'camera': 'handheld camera, natural movement, documentary style', 'lighting': 'natural lighting, soft ambient light, realistic shadows', 'quality': 'photorealistic, ultra realistic, 8k, professional photo', 'negative': 'cartoon, anime, stylized, artificial lighting, oversaturated'},
    'anime': {'camera': 'dynamic anime camera, expressive angles, smooth panning', 'lighting': 'vibrant lighting, cel shaded, studio quality lighting', 'quality': 'anime style, cel shaded, vibrant colors, detailed background, studio quality', 'negative': 'realistic, photorealistic, dark, muted colors, 3d render'},
    'documentary': {'camera': 'steady camera, observational, wide establishing shots', 'lighting': 'natural lighting, available light, realistic', 'quality': 'documentary style, professional photography, realistic texture', 'negative': 'stylized, dramatic, artificial, cinematic, music video'},
    'music video': {'camera': 'dynamic camera movement, fast cuts, tracking shots, crane shots', 'lighting': 'vibrant colors, dynamic lighting, strobe effects, neon', 'quality': 'music video aesthetic, stylized, energetic, high production value', 'negative': 'static, boring, flat, documentary, naturalistic'},
    'poster': {'camera': 'dramatic composition, hero shot, centered framing', 'lighting': 'dramatic lighting, rim light, high contrast, poster quality', 'quality': 'movie poster quality, ultra detailed, professional, striking visual', 'negative': 'amateur, low quality, blurry, flat lighting, boring composition'},
}

NEGATIVE_PROMPT = 'worst quality, inconsistent motion, blurry, jittery, distorted, low resolution, artifacts, static, overexposed, identity drift, deformation, flickering, ghosting, smearing, duplication, mutated proportions, inconsistent clothing, flat colors, desaturated'

def enhance_prompt(prompt, style='cinematic'):
    mod = STYLE_MODIFIERS.get(style, STYLE_MODIFIERS['cinematic'])
    enhanced = f"{prompt}. {mod['camera']}. {mod['lighting']}. {mod['quality']}. Smooth temporal motion, consistent identity throughout. Professional grade output."
    negative = f"{NEGATIVE_PROMPT}, {mod['negative']}"
    return enhanced, negative

def unload_all_models():
    global _pipes
    for key in list(_pipes.keys()):
        del _pipes[key]
    _pipes.clear()
    gc.collect()
    torch.cuda.empty_cache()

def load_sdxl():
    if 'sdxl' not in _pipes:
        unload_all_models()
        from diffusers import StableDiffusionXLPipeline
        print('[Loading SDXL...]')
        _pipes['sdxl'] = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, cache_dir='/kaggle/working/model_cache')
        _pipes['sdxl'].watermark = None
    return _pipes['sdxl']

def load_ltx():
    if 'ltx' not in _pipes:
        unload_all_models()
        from diffusers import LTXPipeline
        print('[Loading LTX-Video...]')
        _pipes['ltx'] = LTXPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.bfloat16, cache_dir='/kaggle/working/model_cache')
        _pipes['ltx'].vae.enable_tiling()
    return _pipes['ltx']

class GenRequest(BaseModel):
    prompt: str
    model: str = 'ltx'
    style: str = 'cinematic'
    num_frames: int = 97
    fps: int = 24
    steps: int = 30
    seed: Optional[int] = None
    enhance: bool = True
    negative_prompt: Optional[str] = None
    width: int = 768
    height: int = 512
    guidance_scale: float = 3.0
    guidance_rescale: float = 0.0

class ImageGenRequest(BaseModel):
    prompt: str
    model: str = 'sdxl'
    image_mode: str = 't2i'
    aspect_ratio: str = '2:3'
    quality: str = 'pro'
    seed: Optional[int] = None
    batch_count: int = 1
    style_preset: str = 'cinematic'
    negative_prompt: str = ''
    width: Optional[int] = None
    height: Optional[int] = None
    guidance_scale: float = 7.5
    steps: Optional[int] = None

@app.get('/api/status')
async def status():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9 if torch.cuda.is_available() else 0
    return {'status': 'online', 'gpu': gpu_name, 'vram_total': f'{vram_total:.1f} GB', 'vram_free': f'{vram_free:.1f} GB', 'models': ['sdxl', 'ltx-video'], 'features': ['prompt-enhancement', 'image-generation', 'video-generation', 'terminal'], 'terminal_token': TERMINAL_TOKEN}

@app.post('/api/image/generate')
async def image_generate(req: ImageGenRequest, bg: BackgroundTasks):
    jid = uuid.uuid4().hex[:12]
    image_status[jid] = {'status': 'processing', 'prompt': req.prompt}
    def run():
        try:
            am = {'1:1': (1024,1024), '16:9': (1344,768), '9:16': (768,1344), '4:3': (1152,896), '3:4': (896,1152), '2:3': (832,1216), '3:2': (1216,832)}
            w, h = am.get(req.aspect_ratio, (1024,1024))
            if req.width: w = req.width
            if req.height: h = req.height
            qs = {'draft': 10, 'standard': 25, 'pro': 40, 'ultra': 60}
            actual_steps = qs.get(req.quality, req.steps or 25)
            enhanced, neg = enhance_prompt(req.prompt, req.style_preset)
            if req.negative_prompt: neg = f'{neg}, {req.negative_prompt}'
            pipe = load_sdxl()
            pipe.to('cuda')
            generator = torch.Generator('cuda').manual_seed(req.seed) if req.seed else None
            images = pipe(prompt=enhanced, negative_prompt=neg, width=w, height=h, num_inference_steps=actual_steps, guidance_scale=req.guidance_scale, generator=generator, num_images_per_prompt=req.batch_count).images
            paths = []
            for img in images:
                path = os.path.join(IMG_DIR, f'img_{uuid.uuid4().hex[:8]}.png')
                img.save(path)
                paths.append(path)
            pipe.to('cpu')
            image_status[jid] = {'status': 'completed', 'prompt': req.prompt, 'output': paths[0] if paths else None, 'images': paths}
            print(f'[Image done: {jid}]')
        except Exception as e:
            image_status[jid] = {'status': 'failed', 'error': str(e), 'traceback': traceback.format_exc()}
    bg.add_task(run)
    return {'job_id': jid, 'status': 'processing'}

@app.get('/api/image/status/{job_id}')
async def image_status_check(job_id: str):
    if job_id not in image_status: return JSONResponse({'error': 'Not found'}, status_code=404)
    return image_status[job_id]

@app.get('/api/image/download/{job_id}')
async def image_download(job_id: str):
    j = image_status.get(job_id)
    if not j or j['status'] != 'completed' or not j.get('output'): return JSONResponse({'error': 'Not ready'}, status_code=400)
    return FileResponse(j['output'], media_type='image/png', filename=f'img_{job_id}.png')

@app.post('/api/generate')
async def generate(req: GenRequest, bg: BackgroundTasks):
    jid = uuid.uuid4().hex[:12]
    generation_status[jid] = {'status': 'processing', 'progress': 0, 'prompt': req.prompt, 'model': req.model}
    def run():
        try:
            if req.enhance:
                enhanced, neg = enhance_prompt(req.prompt, req.style)
            else:
                enhanced = req.prompt
                neg = req.negative_prompt or NEGATIVE_PROMPT
            if req.negative_prompt: neg = f'{neg}, {req.negative_prompt}'
            pipe = load_ltx()
            pipe.to('cuda')
            if req.seed is None: req.seed = int(time.time())
            generator = torch.Generator('cuda').manual_seed(req.seed)
            result = pipe(prompt=enhanced, negative_prompt=neg, width=req.width, height=req.height, num_frames=req.num_frames, num_inference_steps=req.steps, guidance_scale=req.guidance_scale, guidance_rescale=req.guidance_rescale, generator=generator, output_type='pil', decode_timestep=0.05, decode_noise_scale=0.025)
            frames = result.frames
            if isinstance(frames, list): frames = frames[0]
            import imageio
            output_path = os.path.join(VID_DIR, f'vid_{uuid.uuid4().hex[:8]}.mp4')
            frames_np = [np.array(f) for f in frames]
            imageio.mimsave(output_path, frames_np, fps=req.fps, codec='libx264', quality=8, macro_block_size=1)
            pipe.to('cpu')
            generation_status[jid] = {'status': 'complete', 'progress': 1.0, 'prompt': req.prompt, 'output': output_path}
            print(f'[Video done: {jid}]')
        except Exception as e:
            generation_status[jid] = {'status': 'failed', 'prompt': req.prompt, 'error': str(e), 'traceback': traceback.format_exc()}
    bg.add_task(run)
    return {'job_id': jid, 'status': 'processing', 'model': req.model}

@app.get('/api/status/{job_id}')
async def job_status(job_id: str):
    if job_id not in generation_status: return JSONResponse({'error': 'Not found'}, status_code=404)
    return generation_status[job_id]

@app.get('/api/download/{job_id}')
async def download(job_id: str):
    j = generation_status.get(job_id)
    if not j or j['status'] != 'complete' or not j['output']: return JSONResponse({'error': 'Not ready'}, status_code=400)
    return FileResponse(j['output'], media_type='video/mp4', filename=f'video_{job_id}.mp4')

# --- Terminal WebSocket endpoint ---
@app.websocket('/api/terminal/ws')
async def terminal_ws(ws: WebSocket):
    await ws.accept()
    token = ws.query_params.get('token', '')
    if token != TERMINAL_TOKEN:
        await ws.send_json({'type': 'error', 'data': 'Invalid token'})
        await ws.close()
        return
    await ws.send_json({'type': 'ready', 'data': 'Terminal connected'})
    proc = None
    try:
        proc = await asyncio.create_subprocess_exec(
            '/bin/bash', '-i',
            stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.STDOUT,
            env={**os.environ, 'TERM': 'xterm-256color', 'PS1': 'soulillusions@kaggle:~$ '}
        )
        async def read_output():
            while True:
                data = await proc.stdout.read(4096)
                if not data:
                    break
                await ws.send_json({'type': 'output', 'data': data.decode('utf-8', errors='replace')})
        async def write_input():
            while True:
                msg = await ws.receive_text()
                data = json.loads(msg)
                if data.get('type') == 'input':
                    proc.stdin.write(data['data'].encode())
                    await proc.stdin.drain()
                elif data.get('type') == 'resize':
                    pass
        await asyncio.gather(read_output(), write_input())
    except WebSocketDisconnect:
        pass
    except Exception as e:
        try:
            await ws.send_json({'type': 'error', 'data': str(e)})
        except:
            pass
    finally:
        if proc and proc.returncode is None:
            proc.kill()

# One-shot command execution (for Prime Agent / automation)
@app.post('/api/terminal/exec')
async def terminal_exec(request: Request):
    body = await request.json()
    token = body.get('token', '')
    if token != TERMINAL_TOKEN:
        return JSONResponse({'error': 'Invalid token'}, status_code=403)
    cmd = body.get('command', '')
    if not cmd:
        return JSONResponse({'error': 'No command'}, status_code=400)
    proc = await asyncio.create_subprocess_shell(
        cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE
    )
    stdout, stderr = await asyncio.wait_for(proc.communicate(), timeout=60)
    return {'stdout': stdout.decode('utf-8', errors='replace'), 'stderr': stderr.decode('utf-8', errors='replace'), 'returncode': proc.returncode}

print('  Backend code loaded!')

# --- Step 3: Start server ---
print('[3/6] Starting server...')
os.system('pkill -f uvicorn')
os.system('pkill -f cloudflared')
time.sleep(2)

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import urllib.request
for i in range(20):
    time.sleep(2)
    try:
        r = urllib.request.Request('http://localhost:8000/api/status')
        with urllib.request.urlopen(r, timeout=5) as resp:
            d = json.loads(resp.read().decode())
            print(f'  Server OK! GPU: {d.get("gpu")}')
            break
    except:
        print(f'  Waiting... ({(i+1)*2}s)')
else:
    print('ERROR: Server failed to start!')
    raise RuntimeError('Server failed')

# --- Step 4: Start Cloudflare Tunnel ---
print('[4/6] Starting Cloudflare Tunnel...')
cf_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

public_url = None
for i in range(30):
    time.sleep(2)
    chunk = cf_process.stdout.read1(4096).decode('utf-8', errors='ignore')
    urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', chunk)
    if urls:
        public_url = urls[0]
        break

if not public_url:
    print('ERROR: No tunnel URL found!')
    raise RuntimeError('Tunnel failed')

# --- Step 5: Verify and write output file ---
print('[5/6] Verifying connection...')
verified = False
for i in range(10):
    time.sleep(3)
    try:
        r = urllib.request.Request(f'{public_url}/api/status')
        with urllib.request.urlopen(r, timeout=10) as resp:
            d = json.loads(resp.read().decode())
            verified = True
            break
    except:
        print(f'  Verifying... ({(i+1)*3}s)')

# Write tunnel URL + token to output file for kaggle_auto.py to extract
output_info = {'url': public_url, 'terminal_token': TERMINAL_TOKEN, 'gpu': torch.cuda.get_device_name(0), 'verified': verified}
with open('/kaggle/working/tunnel_info.json', 'w') as f:
    json.dump(output_info, f)
print(f'  Tunnel info written to /kaggle/working/tunnel_info.json')

# --- Step 6: Print results ---
print(f'[6/6] Done!')
print(f'\n{"="*60}')
if verified:
    print(f'  SoulIllusions GPU Backend is LIVE!')
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Models: LTX-Video + SDXL')
    print(f'  Terminal: Enabled (token in /api/status)')
else:
    print(f'  URL created but still warming up.')
    print(f'  Wait 10 seconds then try the URL.')
print(f'')
print(f'  PUBLIC URL: {public_url}')
print(f'')
print(f'  Paste this into SoulIllusions Backend URL field!')
print(f'{"="*60}')
print(f'\n  Keep this notebook running - server runs while Kaggle is active.')
print(f'  Terminal tab in SoulIllusions can connect to this backend.')
